# 00 · Escenario y convenciones

**Objetivo.** Documentar la base común del golden model de ISI y CD:

- las unidades y sus conversiones (`C`, `D_si`, `S_si`);
- el escenario de simulación (`Scenario`);
- los parámetros de hardware (`HardwareParams`);
- las convenciones que comparten todos los módulos.

Todo lo que se muestra acá sale de `emulador/config.py`, el mismo código que usan los tests (`tests/test_config.py`).

Referencias: `referencia_guia_ISI_CD.md` §2 y §3; PDF *Fundamentos* §2.4 y Tabla 5; `GUIA_ISI_CD.md` §1 (decisiones 1, 2 y 7) y paso P1.

In [ ]:
%load_ext autoreload
%autoreload 2

import dataclasses

from emulador.config import C, D_si, S_si, HardwareParams, Scenario

## 1. Teoría: señal, muestreo y convenciones

El emulador trabaja sobre la **envolvente compleja en banda base** del campo óptico, muestreada a $f_s$:

$$
x[n] = A(0, nT_s), \qquad f_s = \frac{1}{T_s} = \mathrm{sps}\cdot R_s
$$

donde $R_s$ es la tasa de símbolos y sps la cantidad de muestras por símbolo. La frecuencia $f$ es siempre la desviación respecto de la portadora y recorre $[-f_s/2,\, f_s/2)$.

| Aspecto | Convención |
|---|---|
| Unidades | SI en todo el cálculo interno: $D$ en s/m², $S$ en s/m³, $\lambda$ y $z$ en m. Las unidades de hoja de datos se convierten **solo** con `D_si` / `S_si` |
| Transformada | La de `numpy.fft`: $X[k] = \sum_n x[n]\,e^{-j2\pi kn/L}$ |
| Eje de frecuencias | Siempre `np.fft.fftfreq(L, 1/fs)`; `fftshift` solo para graficar |
| Signo de la CD | **Positivo**: $H_{CD}(f) = \exp\!\left(+j\,\pi D\lambda^2 z f^2/c\right)$, equivalente a $e^{-j\beta(\omega)z}$ con $\beta_2 = -D\lambda^2/(2\pi c)$ |
| Retardos | Cada bloque declara su retardo entero en muestras (`delay_samples`) |
| Aleatoriedad | `np.random.default_rng(seed)`; nunca el estado global de numpy |

**Signo de la CD, en palabras.** Con $D > 0$ (fibra estándar), las frecuencias más altas llegan *antes*: el retardo de grupo $\tau(f) = -D\lambda^2 z f/c$ tiene pendiente negativa. Otros autores usan el signo opuesto, y las dos convenciones son correctas si se usan de forma consistente. Lo importante es que el compensador (tesis de CDC) use la **conjugada** de este $H_{CD}$; si no, duplica la dispersión en vez de anularla. El test de signo del paso B1 fija esta convención en el código.

## 2. Unidades: `C`, `D_si`, `S_si`

`C` es la velocidad de la luz **exacta** (299 792 458 m/s). No se usa `3e8`: con esa aproximación, algunos valores de $R$ de la Tabla 6 del PDF cambian.

Las hojas de datos de fibra dan $D$ en ps/(nm·km) y $S$ en ps/(nm²·km). Los bloques reciben esas unidades (campos `D_ps_nm_km`, `S_ps_nm2_km`) y las convierten a SI con:

$$
1\ \frac{\text{ps}}{\text{nm·km}} = \frac{10^{-12}}{10^{-9}\cdot 10^{3}}\ \frac{\text{s}}{\text{m}^2} = 10^{-6}\ \frac{\text{s}}{\text{m}^2},
\qquad
1\ \frac{\text{ps}}{\text{nm}^2\text{·km}} = \frac{10^{-12}}{10^{-18}\cdot 10^{3}}\ \frac{\text{s}}{\text{m}^3} = 10^{3}\ \frac{\text{s}}{\text{m}^3}
$$

Convertir en un único lugar evita el error más común de este tipo de modelos: un factor $10^{3}$ perdido entre nm, km y m.

In [ ]:
print(f"C              = {C:.0f} m/s")
print(f"D_si(17)       = {D_si(17.0):.2e} s/m^2   (SSMF G.652, Tabla 5 del PDF)")
print(f"S_si(0.057)    = {S_si(0.057):.1f} s/m^3     (pendiente típica de SSMF)")
print(f"D_si(-80)      = {D_si(-80.0):.2e} s/m^2  (fibra compensadora: D negativo es válido)")

## 3. `Scenario`: el canal que se simula

`Scenario` reúne todo lo que depende de la **velocidad del sistema**: $R_s$, sps, $\lambda$ y el roll-off del RRC. De ahí se derivan $f_s$ y $T_s$.

**Por qué un único objeto.** Muchas magnitudes de los bloques dependen de la velocidad:
- el ancho de banda de los perfiles de ISI se da como fracción de $R_s$ (por ejemplo, $B = 0{,}6\,R_s$);
- la longitud del chirp de la CD, $R$, crece con $1/T_s^2$;
- el tamaño de bloque $N$ depende de $R$.

Si cada módulo fijara su propia $f_s$, pasar de 32 a 100 GBd obligaría a tocar todo el código. Con `Scenario`, se cambia una línea.

| Campo | Por defecto | Rango aceptado |
|---|---|---|
| `Rs` | 32 GBd | 1 a 300 GBd |
| `sps` | 2 | 1, 2, 4, 8, 16 (16 solo para análisis fino del instante de muestreo) |
| `wavelength` | 1550 nm | 1530 a 1565 nm (banda C) |
| `rolloff` | 0,1 | 0,05 a 0,5 |

In [ ]:
esc = Scenario()   # escenario de referencia (decisión 1 de §1)
esc

In [ ]:
print(f"fs = {esc.fs/1e9:.0f} GSa/s,  Ts = {esc.Ts*1e12:.3f} ps")
esc.to_dict()   # lo que se guarda en cada meta.json

Cambiar de escenario es crear otro `Scenario`: todo lo derivado se recalcula solo. La tabla muestra cómo crece $f_s$ con la tasa del Proyecto Marco (≥ 100 GBd). Como $R \propto f_s^2$, a 200 GBd la CD necesita bloques mucho más largos (ver §6 de la guía).

In [ ]:
print(f"{'Rs [GBd]':>9} {'sps':>4} {'fs [GSa/s]':>11} {'Ts [ps]':>9}")
for Rs in (32e9, 64e9, 100e9, 200e9):
    e = Scenario(Rs=Rs)
    print(f"{Rs/1e9:>9.0f} {e.sps:>4d} {e.fs/1e9:>11.0f} {e.Ts*1e12:>9.3f}")

### Validación e inmutabilidad

- **Valida los rangos al crearse**, así un valor fuera de rango (por ejemplo, 1310 nm, fuera de la banda C) falla en ese momento y no más adelante con un resultado sin sentido.
- **Es inmutable** (`frozen=True`). Si se pudiera modificar después de crearlo, un bloque podría haber calculado sus coeficientes con un $R_s$ y la señal generarse con otro. Para otro escenario se crea uno nuevo.

In [ ]:
for kwargs in ({"sps": 3}, {"wavelength": 1310e-9}, {"Rs": 500e9}):
    try:
        Scenario(**kwargs)
    except ValueError as exc:
        print(f"Scenario(**{kwargs}) -> ValueError: {exc}")

try:
    esc.Rs = 100e9
except dataclasses.FrozenInstanceError:
    print("esc.Rs = 100e9 -> FrozenInstanceError (usar Scenario(Rs=100e9))")

## 4. `HardwareParams`: dónde se va a implementar

`HardwareParams` describe la **implementación**, no el canal:

| Campo | Significado | Valor actual |
|---|---|---|
| `P` | Muestras por ciclo de reloj de la FFT radix-2² | 1 (provisorio) |
| `f_clk` | Frecuencia de reloj de la FPGA | sin definir |
| `N_max` | Máximo tamaño de bloque del overlap-save | 2048 (provisorio) |
| `provisional` | Si los valores ya están confirmados | `True` hasta el paso C1 |

**Por qué está separado de `Scenario`.** El mismo canal (32 GBd, 100 km) se puede implementar con distinto hardware ($P = 1$ o $P = 8$), y el mismo hardware puede emular distintos canales. Además, hoy el hardware es incierto (decisión 7 de §1: RFSoC sin modelo confirmado) y el escenario no.

**Quién lo va a usar:**
- `cd_N` (paso B1), que elige el $N$ más chico que cumple $N \ge R$ y falla si supera `N_max`;
- la tasa efectiva del bloque de CD con overlap-save, $R_{CD} = P\cdot f_{clk}/2$ (PDF Ec. 31), que se analiza en B4;
- el modelo bit a bit (pasos C1 y C3).

In [ ]:
hw = HardwareParams()
hw

**Ejemplo ilustrativo** (el valor de $f_{clk}$ **no está decidido**): con $f_{clk} = 400$ MHz, la tasa efectiva del bloque de CD para distintos $P$, comparada con los 64 GSa/s del escenario de referencia. Muestra por qué $P$ es el parámetro de compromiso entre velocidad y recursos.

In [ ]:
f_clk_ejemplo = 400e6   # ILUSTRATIVO: no es un valor decidido
for P in (1, 8, 64, 256, 512):
    R_cd = HardwareParams(P=P, f_clk=f_clk_ejemplo).P * f_clk_ejemplo / 2
    print(f"P = {P:>3d}:  R_CD = {R_cd/1e9:7.2f} GSa/s   ({100*R_cd/esc.fs:6.2f} % de fs = {esc.fs/1e9:.0f} GSa/s)")

## 5. Quién usa cada elemento

| Elemento | Qué representa | Quién lo usa |
|---|---|---|
| `C`, `D_si`, `S_si` | Física y unidades | Bloques (convierten los campos de hoja de datos) y `cd/physics.py` (calcula en SI) |
| `Scenario` | El canal: $R_s$, sps, $\lambda$, roll-off → $f_s$, $T_s$ | Perfiles de ISI ($B = B_{rel}\cdot R_s$), CD ($R$, $N$), estímulos (sps, roll-off), receptor, `meta.json` |
| `HardwareParams` | La implementación: $P$, $f_{clk}$, $N_{max}$ | `cd_N`, tasa del bloque de CD, modelo bit a bit |

## 6. Validación

Los mismos criterios que `tests/test_config.py`:

In [ ]:
assert C == 299_792_458.0
assert abs(D_si(17.0) - 1.7e-5) <= 1e-12 * 1.7e-5
assert abs(S_si(0.057) - 57.0) <= 1e-12 * 57.0
assert esc.fs == 64e9 and esc.Ts == 1 / 64e9
assert Scenario(Rs=100e9).fs == 200e9
assert hw.provisional and hw.N_max == 2048
print("Validación OK")

## 7. Conclusiones

- Todo lo que depende de la velocidad del sistema sale de un único `Scenario`: cambiar de 32 a 100 GBd es cambiar una línea.
- Las unidades de hoja de datos se convierten a SI en un único lugar (`D_si`, `S_si`), con `C` exacta.
- El hardware (`HardwareParams`) está separado del canal y sus valores son **provisorios** hasta confirmar el RFSoC (paso C1).
- La convención de signo positiva de la CD queda documentada acá y se fija con un test en el paso B1.

**Pendiente en este notebook:**
- magnitudes derivadas de la CD ($\beta_2$, $\beta_3$, $R$, $L_D$), que se agregan al terminar B1 con `cd_summary`;
- estilo de gráficos del proyecto, que se agrega al terminar P4.2.